In [1]:
from src.sae_ts.inteval.utils import my_multi_criterion_evaluation
from src.sae_ts.steering.evals_util_gemini import my_multi_criterion_evaluation_gemini
import os
import pandas as pd

In [ ]:
base_path = '/Users/z5517269/Desktop/Repos/intsteer/steer_cfgs'
output_path = 'evals.csv'

judges = ["models/gemini-2.5-flash"]
models = ['gemma2', 'gemma2-9b']
criteria = [
    ('anger', 'anger'),
    ('christian_evangelist', 'Christian'),
    ('conspiracy', 'conspiracy'),
    ('french', 'French'),
    ('london', 'London'),
    ('love', 'love'),
    ('praise', 'praise'),
    ('want_to_die', 'want to die'),
    ('wedding', 'wedding')
]
extraction_methods = ['SAE', 'ActSteer']
steering_methods = ['addition', 'rotation']

# ---------------------------------------------------
# Load existing results if present (resume support)
# ---------------------------------------------------
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed = set(
        zip(
            df_existing.judge,
            df_existing.model,
            df_existing.c1,
            df_existing.c2,
            df_existing.extraction_method,
            df_existing.steering_method,
        )
    )
    print(f"Resuming: {len(completed)} evaluations already completed.")
else:
    df_existing = pd.DataFrame()
    completed = set()

# ---------------------------------------------------
# Main evaluation loop
# ---------------------------------------------------
rows_to_append = []

for judge in judges:
    for model in models:
        for c1, c2 in criteria:
            for extraction_method in extraction_methods:
                for steering_method in steering_methods:

                    key = (
                        judge,
                        model,
                        c1,
                        c2,
                        extraction_method,
                        steering_method,
                    )

                    # Skip if already computed
                    if key in completed:
                        continue

                    try:
                        
                        graph_data = my_multi_criterion_evaluation_gemini(
                            f'{base_path}/{model}/{c1}',
                            f'{steering_method}_{c2}_{extraction_method}',
                            model=judge
                        )
                        print(f'Results for {key} is completed')
                        # Enrich metadata
                        graph_data.update({
                            'judge': judge,
                            'model': model,
                            'c1': c1,
                            'c2': c2,
                            'extraction_method': extraction_method,
                            'steering_method': steering_method,
                        })

                        rows_to_append.append(graph_data)

                        # ---------------------------------------------------
                        # Persist immediately (CRASH-SAFE)
                        # ---------------------------------------------------
                        pd.DataFrame([graph_data]).to_csv(
                            output_path,
                            mode='a',
                            header=not os.path.exists(output_path),
                            index=False
                        )

                        completed.add(key)

                    except Exception as e:
                        print(f"[ERROR] {key}: {e}")
                        # Optional: log errors to a separate file
                        continue


Resuming: 162 evaluations already completed.
Submitted 16 batch jobs. Waiting for completion...
Max product: 0.16509633005401234 at scale 100
Results for ('models/gemini-2.5-flash', 'gemma2-9b', 'london', 'London', 'ActSteer', 'rotation') is completed


In [ ]:
base_path = '/Users/z5517269/Desktop/Repos/intsteer/steer_cfgs'
output_path = 'evals.csv'

judges = ["gpt-4o-mini", "gpt-4.1-nano"]
models = ['gemma2', 'gemma2-9b']
criteria = [
    ('anger', 'anger'),
    ('christian_evangelist', 'Christian'),
    ('conspiracy', 'conspiracy'),
    ('french', 'French'),
    ('london', 'London'),
    ('love', 'love'),
    ('praise', 'praise'),
    ('want_to_die', 'want to die'),
    ('wedding', 'wedding')
]
extraction_methods = ['SAE', 'ActSteer']
steering_methods = ['addition', 'rotation']

# ---------------------------------------------------
# Load existing results if present (resume support)
# ---------------------------------------------------
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed = set(
        zip(
            df_existing.judge,
            df_existing.model,
            df_existing.c1,
            df_existing.c2,
            df_existing.extraction_method,
            df_existing.steering_method,
        )
    )
    print(f"Resuming: {len(completed)} evaluations already completed.")
else:
    df_existing = pd.DataFrame()
    completed = set()

# ---------------------------------------------------
# Main evaluation loop
# ---------------------------------------------------
rows_to_append = []

for judge in judges:
    for model in models:
        for c1, c2 in criteria:
            for extraction_method in extraction_methods:
                for steering_method in steering_methods:

                    key = (
                        judge,
                        model,
                        c1,
                        c2,
                        extraction_method,
                        steering_method,
                    )

                    # Skip if already computed
                    if key in completed:
                        continue

                    try:
                        graph_data = my_multi_criterion_evaluation(
                            f'{base_path}/{model}/{c1}',
                            f'{steering_method}_{c2}_{extraction_method}',
                            model=judge
                        )

                        # Enrich metadata
                        graph_data.update({
                            'judge': judge,
                            'model': model,
                            'c1': c1,
                            'c2': c2,
                            'extraction_method': extraction_method,
                            'steering_method': steering_method,
                        })

                        rows_to_append.append(graph_data)

                        # ---------------------------------------------------
                        # Persist immediately (CRASH-SAFE)
                        # ---------------------------------------------------
                        pd.DataFrame([graph_data]).to_csv(
                            output_path,
                            mode='a',
                            header=not os.path.exists(output_path),
                            index=False
                        )

                        completed.add(key)

                    except Exception as e:
                        print(f"[ERROR] {key}: {e}")
                        # Optional: log errors to a separate file
                        continue


In [5]:
import pandas as pd
import krippendorff
import numpy as np
import pingouin as pg
df = pd.read_csv('evals.csv')
df = df[df.judge.isin(['gpt-4o-mini', 'models/gemini-2.5-flash'])]

df['marks'] = df['result'].apply(lambda item:eval(item)['max_product'])
df['marks_z'] = df.groupby('judge')['marks'].transform(
    lambda x: (x - x.mean()) / x.std(ddof=0)
)
df_gpt = df[df.judge=='gpt-4o-mini']
df_gmn = df[df.judge=='models/gemini-2.5-flash']
merged = df_gpt.merge(df_gmn, on=['path', 'method', 'steering_goal_name'], how='inner')
data = merged[['marks_x', 'marks_y']].to_numpy().T
alpha = krippendorff.alpha(
    reliability_data=data,
    level_of_measurement='interval'
)
print('alpha is: ', alpha)

data_z = merged[['marks_z_x', 'marks_z_y']].to_numpy().T

alpha_z = krippendorff.alpha(
    reliability_data=data_z,
    level_of_measurement='interval'
)

print('Krippendorff alpha (z-scored): ', alpha_z)

mad = np.mean(np.abs(merged['marks_x'] - merged['marks_y']))
print('mean absolute difference: ', mad)
correlation = merged[['marks_x', 'marks_y']].corr().iloc[0,1]
print('Corr: ', correlation)


df['target'] = df.apply(lambda item: ' '.join([item['path'], item['method'], item['steering_goal_name']]), axis=1)
# df[['target', 'judge', 'marks']]
print(df.groupby('judge')['marks'].describe())
icc = pg.intraclass_corr(
    data=df,
    targets='target',
    raters='judge',
    ratings='marks',
    nan_policy='omit'
)

icc[icc['Type'] == 'ICC3']

alpha is:  0.235026983217275
Krippendorff alpha (z-scored):  0.846553882800243
mean absolute difference:  0.18416995505907913
Corr:  0.8454808330296153
                         count      mean       std       min       25%  \
judge                                                                    
gpt-4o-mini               72.0  0.220682  0.107196  0.013158  0.136068   
models/gemini-2.5-flash   72.0  0.402852  0.161939  0.025085  0.311611   

                              50%       75%       max  
judge                                                  
gpt-4o-mini              0.219304  0.299390  0.503499  
models/gemini-2.5-flash  0.388785  0.533482  0.725554  


,Type,Description,ICC,F,df1,df2,pval,CI95%
2,ICC3,Single fixed raters,0.778301,8.021227,71,71,2.683071e-16,"[0.67, 0.86]"


In [12]:
import pandas as pd

df = pd.read_csv('evals.csv')
df = df[df.judge.isin(['gpt-4o-mini', 'models/gemini-2.5-flash'])]
df['marks'] = df['result'].apply(lambda item:eval(item)['max_product'])
df.groupby(['judge', 'model', 'extraction_method', 'steering_method'])['marks'].describe()


count  \
judge                   model     extraction_method steering_method          
gpt-4o-mini             gemma2    ActSteer          addition           9.0   
                                                    rotation           9.0   
                                  SAE               addition           9.0   
                                                    rotation           9.0   
                        gemma2-9b ActSteer          addition           9.0   
                                                    rotation           9.0   
                                  SAE               addition           9.0   
                                                    rotation           9.0   
models/gemini-2.5-flash gemma2    ActSteer          addition           9.0   
                                                    rotation           9.0   
                                  SAE               addition           9.0   
                                                    rotation           9.0   
                        gemma2-9b ActSteer          addition           9.0   
                                                    rotation           9.0   
                                  SAE               addition           9.0   
                                                    rotation           9.0   

                                                                         mean  \
judge                   model     extraction_method steering_method             
gpt-4o-mini             gemma2    ActSteer          addition         0.219230   
                                                    rotation         0.275462   
                                  SAE               addition         0.128083   
                                                    rotation         0.164069   
                        gemma2-9b ActSteer          addition         0.254793   
                                                    rotation         0.291274   
                                  SAE               addition         0.220216   
                                                    rotation         0.212326   
models/gemini-2.5-flash gemma2    ActSteer          addition         0.396504   
                                                    rotation         0.464246   
                                  SAE               addition         0.263540   
                                                    rotation         0.320769   
                        gemma2-9b ActSteer          addition         0.454749   
                                                    rotation         0.516108   
                                  SAE               addition         0.404632   
                                                    rotation         0.402269   

                                                                          std  \
judge                   model     extraction_method steering_method             
gpt-4o-mini             gemma2    ActSteer          addition         0.106142   
                                                    rotation         0.101395   
                                  SAE               addition         0.092680   
                                                    rotation         0.063695   
                        gemma2-9b ActSteer          addition         0.111425   
                                                    rotation         0.124716   
                                  SAE               addition         0.096419   
                                                    rotation         0.083060   
models/gemini-2.5-flash gemma2    ActSteer          addition         0.171847   
                                                    rotation         0.122160   
                                  SAE               addition         0.161032   
                                                    rotation         0.148404   
                        gemma2-9b ActSteer          addition         0.

In [1]:
import pandas as pd

df = pd.read_csv('evals.csv')
df = df[df.judge.isin(['gpt-4o-mini', 'models/gemini-2.5-flash'])]
df['marks'] = df['result'].apply(lambda item:eval(item)['max_product'])
df['best_scale'] = df['result'].apply(lambda item:eval(item)['scale_at_max'])
print(df[(df.judge=='gpt-4o-mini')&(df.model=='gemma2')][['c1', 'marks', 'best_scale']].head(50))



                      c1     marks  best_scale
0                  anger  0.074050          60
1                  anger  0.117808          60
2                  anger  0.178429         120
3                  anger  0.188292          80
4   christian_evangelist  0.090707          60
5   christian_evangelist  0.064756          40
6   christian_evangelist  0.350670          80
7   christian_evangelist  0.351579         100
8             conspiracy  0.221309         140
9             conspiracy  0.247412         100
10            conspiracy  0.357307         140
11            conspiracy  0.363972         120
12                french  0.061569          20
13                french  0.093379          20
14                french  0.280092          20
15                french  0.306320          20
16                london  0.013158          80
17                london  0.176922         100
18                london  0.061919         100
19                london  0.249661         100
20           

In [12]:
df.iloc[18]

path                     /Users/z5517269/Desktop/Repos/intsteer/steer_c...
method                                            addition_London_ActSteer
steering_goal_name                                                  London
scales                   [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 2...
avg_coherence            [0.638454861111111, 0.640625, 0.60763888888888...
avg_score                [0.0, 0.015190972222222222, 0.0095486111111111...
product                  [0.0, 0.00973171657986111, 0.00580210744598765...
individual_scores        [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...
individual_coherences    [[0.6666666666666666, 0.8888888888888888, 0.77...
individual_products      [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...
result                   {'path': '/Users/z5517269/Desktop/Repos/intste...
judge                                                          gpt-4o-mini
model                                                               gemma2
c1                       

In [13]:
df.iloc[19]

path                     /Users/z5517269/Desktop/Repos/intsteer/steer_c...
method                                            rotation_London_ActSteer
steering_goal_name                                                  London
scales                   [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 2...
avg_coherence            [0.6397569444444444, 0.6176215277777778, 0.620...
avg_score                [0.0, 0.013888888888888888, 0.0542534722222222...
product                  [0.0, 0.008578076774691358, 0.0336729449990354...
individual_scores        [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...
individual_coherences    [[0.6666666666666666, 0.7777777777777778, 0.77...
individual_products      [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...
result                   {'path': '/Users/z5517269/Desktop/Repos/intste...
judge                                                          gpt-4o-mini
model                                                               gemma2
c1                       